<!-- nav -->
[← 8.16 · SystemVerilog Constraints](08_16_SystemVerilog_Constraints.ipynb) · [Home](../../README.md) · [9 · Constraints as data →](09_Constraints_As_Data.ipynb)

# 8.17 · SystemVerilog Preprocessor

Verilog's and SystemVerilog's compiler directives (IEEE 1800 clause 22) are written among the code they direct: a macro
is defined, `` `define WIDTH 8 ``, and used where an expression stands, `` `WIDTH - 1 ``; a conditional directive keeps
code or drops it, `` `ifdef SIM ... `endif ``. Tutorial 8 showed a `` `define `` (08.3). This page builds the dialect
that models them, `SystemVerilog Preprocessor`, from a macro usage to the conditional directives, each in the builders'
fluent calls beside the text SystemVerilog writes, the comments kept among them too. Then what is done with them: held
among an expression's terms and a constraint block's items, validated, with one refused; never expanded or simplified,
each kind left in the residual, named; and the editions that have them, every one of Verilog's and SystemVerilog's.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `name` and
`literal` write a name and a literal of SystemVerilog's `Expressions` (tutorials 3 and 4), and `spelled` an identifier,
a macro's name; `refused` gives what a simplifier says of an expression, its value where its residual is a literal, or why it has none, as
tutorial 8's does.

In [1]:
from mbse.Expressions.SystemVerilog import Expressions as SystemVerilog, Preprocessor as SVPreprocessor
from mbse.Expressions.Framework import Standards, Symbolics, Terms
from toolkit import dialects

store = dialects()
S = store.Expressions
Preprocessor = SVPreprocessor.Expressions.DIALECT  # the dialect, as the framework reads it
name = lambda spelling: lambda d: d.nameexpression(lambda n: n.name(lambda a: a.identifier(lambda i: i.spelling(spelling))))
literal = lambda value: lambda d: d.literal(lambda l: l.value(lambda v: v.as_scalar(value)))
spelled = lambda spelling: lambda i: i.spelling(spelling)  # an identifier, of its spelling
x = name("x")



def refused(simplify, expression):
    """What `simplify` says of `expression`: its value, a simplification's where its residual is a literal, or why it has
    none (`NotEvaluated`, a `ValueError`)."""
    try:
        result = simplify(expression)
        return str(result.evaluate() if hasattr(result, "evaluate") else result)
    except ValueError as error:
        return str(error)

<a id="T08.17.1"></a>
## 08.17.1 · A macro usage, where an expression stands

```text
`WIDTH        `MAX(a, b)        `WIDTH - 1
```

A macro usage is a `macrousage`: its name, an identifier, and its arguments, kept as the text they are written in, since
a macro's arguments are text the preprocessor substitutes, not terms. It stands wherever an expression may: every one of
the family's `Term`s lists it, `Expressions`' among them, so `` `WIDTH - 1 `` is `Expressions`' own binary expression,
its left operand a macro usage:

In [2]:
with store.transaction():
    usages = [S.SystemVerilog.Preprocessor.Expression(lambda e: e.root(lambda u: u.macrousage(lambda m: m.name(spelled("WIDTH"))))),
              S.SystemVerilog.Preprocessor.Expression(lambda e: e.root(
                  lambda u: u.macrousage(lambda m: m.name(spelled("MAX")).arguments("a, b"))))]
    less = S.SystemVerilog.Expressions.Expression(lambda e: e.root(lambda d: d.binaryexpression(lambda b: b
        .left(lambda d: d.macrousage(lambda m: m.name(spelled("WIDTH")))).operator("-").right(literal(1)))))
for expression in [*usages, less]:
    form = Terms.form(Terms.term(expression))
    print(f"{SystemVerilog.Text.ToText(expression):11}", f"{form.kind:16}",
          ", ".join(f"{Terms.kind_of(t)[0]} of {Preprocessor.owner(t).name()}" for t in form.arguments) or "-")
assert [SystemVerilog.Text.ToText(e) for e in [*usages, less]] == ["`WIDTH", "`MAX(a, b)", "`WIDTH - 1"]

`WIDTH      macrousage       -
`MAX(a, b)  macrousage       -
`WIDTH - 1  binaryexpression macrousage of SystemVerilog Preprocessor, literal of SystemVerilog


<a id="T08.17.2"></a>
## 08.17.2 · Directives

```text
`define WIDTH 8        `define MAX(a, b) ((a) > (b) ? (a) : (b))        `undef WIDTH
`include "defs.svh"    `timescale 1ns / 1ps    `default_nettype none    `resetall
```

Each directive is a kind of its own: `definedirective` (its name, whether it is function-like, its parameters, and its
body, kept as the text it is, since a macro's body is text until it is substituted), `undefdirective`,
`includedirective`, `timescaledirective` and `defaultnettypedirective`. A directive the dialect has no kind for is an
`otherdirective`, kept as its text:

In [3]:
with store.transaction():
    directives = [S.SystemVerilog.Preprocessor.Expression(lambda e, root=root: e.root(root)) for root in [
        lambda u: u.definedirective(lambda d: d.name(spelled("WIDTH")).body("8")),
        lambda u: u.definedirective(lambda d: d.name(spelled("MAX")).function_like(True)
            .parameters([spelled("a"), spelled("b")]).body("((a) > (b) ? (a) : (b))")),
        lambda u: u.undefdirective(lambda d: d.name(spelled("WIDTH"))),
        lambda u: u.includedirective(lambda d: d.path("defs.svh")),
        lambda u: u.timescaledirective(lambda d: d.unit("1ns").precision("1ps")),
        lambda u: u.defaultnettypedirective(lambda d: d.net_type("none")),
        lambda u: u.otherdirective(lambda d: d.text("`resetall")),
    ]]
for directive in directives:
    print(f"{SystemVerilog.Text.ToText(directive):43}", Terms.kind_of(Terms.term(directive))[0])
assert [SystemVerilog.Text.ToText(d) for d in directives] == [
    "`define WIDTH 8", "`define MAX(a, b) ((a) > (b) ? (a) : (b))", "`undef WIDTH", '`include "defs.svh"',
    "`timescale 1ns / 1ps", "`default_nettype none", "`resetall"]

`define WIDTH 8                             definedirective
`define MAX(a, b) ((a) > (b) ? (a) : (b))   definedirective
`undef WIDTH                                undefdirective
`include "defs.svh"                         includedirective
`timescale 1ns / 1ps                        timescaledirective
`default_nettype none                       defaultnettypedirective
`resetall                                   otherdirective


<a id="T08.17.3"></a>
## 08.17.3 · Conditional directives

```text
`ifdef SIM            `ifdef (FAST && !SIM)
x < 4;                x < 16;
`else                 `elsif SIM
x < 8;                x < 4;
`endif                `endif
```

A conditional directive, `ifdefdirective`, keeps its items where its macro is defined (`` `ifndef ``, `negated`, where
it is not), its `` `elsif `` branches, `ElsifDirective` helpers, where theirs is, and its alternative, after `` `else ``,
where none is. IEEE 1800-2023 lets it test a condition of macros' names, in parentheses, in place of one name. Its items
are what it keeps: here constraints, the `Term` of `SystemVerilog Constraints`, which lists the directives in turn,
so that a constraint block may hold them. The printer writes each directive at the start of a line of its own:

In [4]:
with store.transaction():
    simple_ifdef = S.SystemVerilog.Preprocessor.Expression(lambda e: e.root(lambda u: u.ifdefdirective(lambda d: d.name(spelled("SIM"))
        .items([lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.binaryexpression(lambda b: b.left(x).operator("<").right(literal(4)))))]).alternative([lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.binaryexpression(lambda b: b.left(x).operator("<").right(literal(8)))))]).has_else(True))))
condition = lambda d: d.parenthesizedexpression(lambda p: p.expression(lambda d: d.binaryexpression(lambda b: b.left(name("FAST")).operator("&&").right(lambda n: n.unaryexpression(lambda u: u.operator("!").operand(name("SIM")))))))
with store.transaction():
    chosen = S.SystemVerilog.Preprocessor.Expression(lambda e: e.root(lambda u: u.ifdefdirective(lambda d: d.condition(condition)
        .items([lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.binaryexpression(lambda b: b.left(x).operator("<").right(literal(16)))))])
        .branches([lambda b: b.name(spelled("SIM")).items([lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.binaryexpression(lambda b: b.left(x).operator("<").right(literal(4)))))])]))))
for directive in (simple_ifdef, chosen):
    print(SystemVerilog.Text.ToText(directive))
    print("--", "free:", ", ".join(sorted(Symbolics.free(directive, Preprocessor))))
assert SystemVerilog.Text.ToText(chosen) == "`ifdef (FAST && !SIM)\nx < 16;\n`elsif SIM\nx < 4;\n`endif"

`ifdef SIM
x < 4;
`else
x < 8;
`endif
-- free: x
`ifdef (FAST && !SIM)
x < 16;
`elsif SIM
x < 4;
`endif
-- free: x


`FAST` and `SIM` are macros' names, not values: a conditional directive binds them within its condition alone, so
`Symbolics.free` gives only `x`, the name its items read, which the scope must bind as anywhere else. Where a reader
keeps the text a conditional drops without reading it, that text is a `disabledtext`, as written.

<a id="T08.17.4"></a>
## 08.17.4 · Among constraints, with comments

```text
p.randomize() with { // at least one
x > 0;
`ifdef SIM
x < 4;
`endif
}
```

Since the Constraints dialect's `Term` lists the directives and the dialect's `comment`, `randomize() with` holds them
among its constraints, as mbse-programs reads them from source. The printer lays them out as a program is laid out, a
line comment ending its line and a directive on lines of its own:

In [5]:
with store.transaction():
    randomized = S.SystemVerilog.Expressions.Expression(lambda e: e.root(lambda u: u.randomizewithexpression(lambda r: r
        .call(lambda c: c.callexpression(lambda m: m.callee(lambda d: d.memberexpression(lambda b: b
            .value(name("p")).member(spelled("randomize")))).arguments([])))
        .items([lambda c: c.comment(lambda t: t.text(" at least one")), lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.binaryexpression(lambda b: b.left(x).operator(">").right(literal(0))))),
                lambda c: c.ifdefdirective(lambda d: d.name(spelled("SIM")).items([lambda c: c.expressionconstraint(lambda e: e.expression(lambda d: d.binaryexpression(lambda b: b.left(x).operator("<").right(literal(4)))))]))]))))
print(SystemVerilog.Text.ToText(randomized))
items = Terms.form(Terms.term(randomized)).arguments[1:]
print("--", ", ".join(f"{Terms.kind_of(t)[0]} of {Preprocessor.owner(t).name()}" for t in items))
assert SystemVerilog.Text.ToText(randomized) == "p.randomize() with { // at least one\nx > 0;\n`ifdef SIM\nx < 4;\n`endif\n}"

p.randomize() with { // at least one
x > 0;
`ifdef SIM
x < 4;
`endif
}
-- comment of SystemVerilog Preprocessor, expressionconstraint of SystemVerilog Constraints, ifdefdirective of SystemVerilog Preprocessor


<a id="T08.17.5"></a>
## 08.17.5 · Validated, and one refused

`validate` checks each kind's own rules, with mbse-programs' messages: a directive's name an identifier, a default net
type one IEEE 1800 names, a conditional directive's name or its condition, not both. A condition in place of a name is
written in parentheses, so one without them is refused:

In [6]:
with store.transaction():
    checked = [simple_ifdef, *(S.SystemVerilog.Preprocessor.Expression(lambda e, root=root: e.root(root)) for root in [
        lambda u: u.defaultnettypedirective(lambda d: d.net_type("wirex")),
        lambda u: u.ifdefdirective(lambda d: d
            .condition(lambda d: d.binaryexpression(lambda b: b.left(name("FAST")).operator("&&").right(name("SIM"))))
            .items([lambda c: c.expressionconstraint(lambda e: e.expression(
                lambda d: d.binaryexpression(lambda b: b.left(x).operator("<").right(literal(16)))))])),
    ])]
for directive in checked:
    print(SystemVerilog.Text.ToText(directive).split("\n")[0].ljust(24), "|", "; ".join(Preprocessor.validate(directive, {"x"})) or "valid")
assert Preprocessor.validate(checked[2], {"x"}) == ["an IfdefDirective's condition is in parentheses"]

`ifdef SIM               | valid
`default_nettype wirex   | DefaultNettypeDirective.net_type cannot be 'wirex'
`ifdef FAST && SIM       | an IfdefDirective's condition is in parentheses


<a id="T08.17.6"></a>
## 08.17.6 · Never expanded, never simplified

No macro is expanded and no directive is run: the dialect models the text a preprocessor reads, not what it makes of
it. SystemVerilog's rules fold none of its kinds: each stays in the residual, named (stage 6, "Q11"), held alone or as an expression's operand:
`` `WIDTH - 1 `` has no value until a preprocessor gives `` `WIDTH `` one:

In [7]:
evaluate = SystemVerilog.Simplifiers.OfAny
for expression in [usages[0], directives[0], simple_ifdef, less]:
    print(f"{SystemVerilog.Text.ToText(expression).split(chr(10))[0]:16}", refused(lambda e: evaluate(e, {"x": 1}), expression))

`WIDTH           SystemVerilog Preprocessor macrousage is not evaluated
`define WIDTH 8  SystemVerilog Preprocessor definedirective is not evaluated
`ifdef SIM       SystemVerilog Preprocessor ifdefdirective is not evaluated
`WIDTH - 1       SystemVerilog Preprocessor macrousage is not evaluated


<a id="T08.17.7"></a>
## 08.17.7 · Its editions are data

Compiler directives are Verilog's as much as SystemVerilog's, so the dialect covers all eight editions, IEEE Std
1364-1995 to 1800-2023, each with every one of its kinds
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):

In [8]:
singleton = store.Instances.Expressions.SystemVerilog.Preprocessor
kinds = {branch.schema: branch.name for branch in store.registered(Preprocessor.union).branches}
for designation, edition in Standards.standards(singleton).items():
    print(f"{designation:19}", f"{Standards.tags(edition)[1]:19}", len(Standards.kinds(edition)), "kinds")
print("each:", ", ".join(kinds[k] for k in Standards.kinds(Standards.find(singleton, "Verilog-1995"))))

IEEE Std 1364-1995  Verilog-1995        10 kinds
IEEE Std 1364-2001  Verilog-2001        10 kinds
IEEE Std 1364-2005  Verilog-2005        10 kinds
IEEE Std 1800-2005  SystemVerilog-2005  10 kinds
IEEE Std 1800-2009  SystemVerilog-2009  10 kinds
IEEE Std 1800-2012  SystemVerilog-2012  10 kinds
IEEE Std 1800-2017  SystemVerilog-2017  10 kinds
IEEE Std 1800-2023  SystemVerilog-2023  10 kinds
each: macrousage, includedirective, definedirective, undefdirective, timescaledirective, defaultnettypedirective, ifdefdirective, disabledtext, otherdirective, comment


An edition records the kinds it has, not each of a kind's forms: a conditional directive's condition in parentheses is
IEEE 1800-2023's, but `ifdefdirective` is in every edition's.

## What you learned

- **Directives, macro usages and comments are terms of `SystemVerilog Preprocessor`**, each in the builders' calls
  beside its text; a macro's body and arguments are text, kept as written.
- **A macro usage stands wherever an expression does**, and the directives and comments among a constraint block's
  items, which a conditional directive's items are in turn.
- **A conditional directive's condition names macros**, bound within it alone; it is validated with mbse-programs'
  messages, a condition without parentheses refused.
- **Nothing is expanded or simplified**: each kind is left in the residual, named; its editions are all of Verilog's and
  SystemVerilog's.

That ends SystemVerilog's sublanguages. Next, Part II puts expressions to work as constraints, starting with case
study 9.

<!-- nav -->
[← 8.16 · SystemVerilog Constraints](08_16_SystemVerilog_Constraints.ipynb) · [Home](../../README.md) · [9 · Constraints as data →](09_Constraints_As_Data.ipynb)